# 🎙️ Chichewa Automatic Speech Recognition
## Fine-tuning Whisper-tiny on Mozilla Common Voice data

This notebook fine-tunes OpenAI's Whisper-tiny model to transcribe spoken Chichewa into text.

**What is Automatic Speech Recognition (ASR)?**
ASR converts spoken audio into written text. You speak — the model types. It is the technology behind voice assistants, transcription services, and accessibility tools.

**What we use:**
- **Model:** `openai/whisper-tiny` — only 39MB, already trained on multilingual audio including African languages. Fine-tuning builds on this existing knowledge.
- **Data:** Mozilla Common Voice 17.0 — a community-collected dataset of people reading Chichewa sentences aloud. Config code: `ny` (Nyanja/Chichewa).
- **Metric:** Word Error Rate (WER) — the percentage of words the model gets wrong. Lower is better. 0% = perfect, 100% = completely wrong.

**Session plan:**
1. Set up and download everything (Cell 1)
2. Explore the dataset — listen to samples (Cell 2)
3. Prepare audio features for training (Cell 3)
4. Fine-tune Whisper-tiny — 1 epoch on a small subset (Cell 4)
5. Evaluate and run inference on new audio (Cell 5)
6. Experiment with more data and epochs (Cell 6)

---

**Before you start — you need a HuggingFace account and token:**
Common Voice requires authentication. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens), create a token with **Read** access, and paste it into Cell 1.

**Install dependencies (run once in terminal):**
```bash
pip install transformers datasets torchaudio librosa soundfile jiwer accelerate
```

## Cell 1 — Setup & Download

This cell downloads everything we need and saves it locally so the rest of the notebook runs without internet.

**Why a HuggingFace token?**
Mozilla Common Voice requires you to agree to their terms of use before downloading. Logging in with your HuggingFace token confirms you have agreed. Go to [huggingface.co/datasets/mozilla-foundation/common_voice_17_0](https://huggingface.co/datasets/mozilla-foundation/common_voice_17_0) and click **Agree and access repository** first.

**What gets downloaded:**
- Common Voice Chichewa audio + transcripts — about 1-2GB. This takes 5-10 minutes.
- Whisper-tiny model weights — about 39MB, very fast.
- Whisper feature extractor and tokenizer — handles converting raw audio to model inputs.

> **Run this cell first and wait for it to complete before running anything else.**

In [ ]:
# ── CELL 1: Setup & Download ──────────────────────────────────────────────

import os
from datasets import load_dataset, DatasetDict
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# ── Configuration ─────────────────────────────────────────────────────────
HF_TOKEN      = ""              # <-- paste your HuggingFace token here
MODEL_NAME    = "openai/whisper-tiny"
DATASET_NAME  = "mozilla-foundation/common_voice_17_0"
LANGUAGE      = "ny"            # Chichewa (Nyanja)
TASK          = "transcribe"    # we want the model to transcribe, not translate
SAVE_DIR      = "./asr_model"
DATA_DIR      = "./asr_data"
SAMPLE_RATE   = 16000           # Whisper expects 16kHz audio

# ── Login to HuggingFace ──────────────────────────────────────────────────
if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN is empty. Go to huggingface.co/settings/tokens, "
        "create a Read token, and paste it above."
    )

from huggingface_hub import login
login(token=HF_TOKEN)
print("Logged in to HuggingFace.")

# ── Download Common Voice Chichewa ────────────────────────────────────────
print(f"\nDownloading Common Voice Chichewa (~1-2GB). This takes 5-10 minutes...")
common_voice = DatasetDict()
for split in ["train", "validation", "test"]:
    common_voice[split] = load_dataset(
        DATASET_NAME, LANGUAGE,
        split=split,
        token=HF_TOKEN,
    )
    print(f"  {split}: {len(common_voice[split])} examples")

# Remove columns we don't need
cols_to_remove = ["accent", "age", "client_id", "down_votes",
                  "gender", "locale", "path", "segment", "up_votes", "variant"]
common_voice = common_voice.remove_columns(
    [c for c in cols_to_remove if c in common_voice["train"].column_names]
)
print(f"  Columns kept: {common_voice['train'].column_names}")

# Save locally
os.makedirs(DATA_DIR, exist_ok=True)
common_voice.save_to_disk(DATA_DIR)
print(f"\nDataset saved to {DATA_DIR}/")

# ── Download Whisper processor and model ──────────────────────────────────
print(f"\nDownloading Whisper processor...")
processor = WhisperProcessor.from_pretrained(
    MODEL_NAME, language="Chichewa", task=TASK
)
processor.save_pretrained("./whisper_processor")
print("Processor saved to ./whisper_processor/")

print(f"\nDownloading Whisper-tiny model weights (~39MB)...")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.save_pretrained("./whisper_base")
print("Model saved to ./whisper_base/")

print("\n✅ Setup complete — everything is now available offline.")

## Cell 2 — Explore the Dataset

Before training, let's look at what the data contains.

**What is in the Common Voice dataset?**
Each example has:
- `audio` — a dictionary containing the raw audio waveform, the sample rate, and the file path
- `sentence` — the Chichewa text that was read aloud

**What is a sample rate?**
Sample rate is how many audio measurements are taken per second. Whisper expects 16,000 samples per second (16kHz). Common Voice records at a different rate, so we resample during preprocessing.

**What is a waveform?**
Audio is stored as a sequence of numbers representing the amplitude (loudness) of the sound at each moment in time. The model learns to read these numbers and predict the text that was spoken.

In [ ]:
# ── CELL 2: Explore the Dataset ───────────────────────────────────────────

from datasets import load_from_disk

# Load from local disk (no internet needed after Cell 1)
common_voice = load_from_disk(DATA_DIR)

print("Dataset splits:")
for split, ds in common_voice.items():
    print(f"  {split}: {len(ds)} examples")

print("\nSample sentences from training set:")
print("-" * 60)
for i in range(5):
    example = common_voice["train"][i]
    audio = example["audio"]
    duration = len(audio["array"]) / audio["sampling_rate"]
    print(f"\nExample {i+1}:")
    print(f"  Text:        {example['sentence']}")
    print(f"  Duration:    {duration:.1f} seconds")
    print(f"  Sample rate: {audio['sampling_rate']} Hz")
    print(f"  Audio shape: {len(audio['array'])} samples")

# Show total audio duration
print("\n" + "-" * 60)
total_seconds = sum(
    len(ex["audio"]["array"]) / ex["audio"]["sampling_rate"]
    for ex in common_voice["train"]
)
print(f"Total training audio: {total_seconds/60:.1f} minutes")
print(f"Average duration per clip: {total_seconds/len(common_voice['train']):.1f} seconds")

## Cell 3 — Prepare Audio Features

Raw audio waveforms cannot be fed directly to Whisper. We need to convert them into a format the model understands. This involves two steps:

**Step 1 — Resample to 16kHz:**
Common Voice records at 48kHz. Whisper was trained on 16kHz audio. We use the `cast_column` method to automatically resample every audio file to 16kHz when it is loaded.

**Step 2 — Extract log-mel spectrograms:**
Instead of feeding raw waveform numbers to the model, we convert the audio into a log-mel spectrogram — a 2D image of the sound that shows which frequencies are present at each moment in time. This is what the model actually reads.

The `WhisperProcessor` handles both steps automatically. It also converts the text transcripts into token IDs (numbers) that the model can learn to predict.

**Data collator:**
During training, examples are grouped into batches. Each audio clip has a different length, so we need to pad shorter ones and handle the labels carefully. The `DataCollatorSpeechSeq2SeqWithPadding` class handles this — it pads audio features to the same length and replaces padding tokens in the labels with -100 so the model ignores them during loss calculation.

In [ ]:
# ── CELL 3: Prepare Audio Features ────────────────────────────────────────

import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from datasets import load_from_disk, Audio
from transformers import WhisperProcessor

common_voice = load_from_disk(DATA_DIR)
processor = WhisperProcessor.from_pretrained(
    "./whisper_processor", language="Chichewa", task=TASK
)

# ── Step 1: Resample all audio to 16kHz ──────────────────────────────────
print("Resampling audio to 16kHz...")
common_voice = common_voice.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
print(f"  Audio will be resampled to {SAMPLE_RATE}Hz on load")

# ── Step 2: Feature extraction function ──────────────────────────────────
def prepare_dataset(batch):
    """
    Converts raw audio to log-mel spectrograms and text to token IDs.
    Called on every example in the dataset.
    """
    audio = batch["audio"]

    # Extract log-mel spectrogram from audio waveform
    # input_features shape: (80 mel bins, 3000 time frames) = 30 seconds max
    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    # Convert text transcript to token IDs
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

# Apply to full dataset
print("Extracting features (this may take a few minutes)...")
common_voice = common_voice.map(
    prepare_dataset,
    remove_columns=common_voice["train"].column_names,
    num_proc=1,
)
print(f"Done. Columns: {common_voice['train'].column_names}")

# ── Data collator ─────────────────────────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """
    Pads audio features and labels to the same length within each batch.
    Labels get -100 for padding positions so the model ignores them.
    """
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Pad input features
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        # Pad labels and replace padding with -100
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Remove beginning-of-sequence token if present
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("\nData collator ready.")
print(f"Training examples: {len(common_voice['train'])}")

## Cell 4 — Fine-tune Whisper-tiny (Live Demo)

This is the training cell. We take Whisper-tiny — already trained on multilingual speech — and fine-tune it specifically for Chichewa.

**What fine-tuning does for ASR:**
Whisper-tiny already knows what speech sounds like in many languages. Fine-tuning teaches it the specific sounds, words, and patterns of Chichewa. With even a small amount of data, the model can improve significantly because it is building on existing speech knowledge.

**Training parameters:**
- `TRAIN_SUBSET` — how many examples to use for the live demo (50 is fast, full dataset is better)
- `NUM_EPOCHS` — how many passes through the data
- `LEARNING_RATE` — how big each gradient descent step is (1e-5 is conservative for Whisper)
- `BATCH_SIZE` — examples per step (4 is safe for CPU with 16GB RAM for audio data)

**Expected time on CPU:**
- 50 examples, 1 epoch ≈ 10-20 minutes
- Full dataset, 3 epochs ≈ several hours

**What to watch:**
- Loss decreasing means the model is learning
- WER (Word Error Rate) on the validation set after each epoch — lower is better

In [ ]:
# ── CELL 4: Fine-tune Whisper-tiny ────────────────────────────────────────

# ── Training configuration ────────────────────────────────────────────────
TRAIN_SUBSET  = 50       # examples for live demo (use None for full dataset)
NUM_EPOCHS    = 1        # training passes
LEARNING_RATE = 1e-5     # conservative for Whisper fine-tuning
BATCH_SIZE    = 4        # safe for CPU with 16GB RAM

import numpy as np
import evaluate
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

# Load WER metric
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    """
    Computes Word Error Rate (WER) on the validation set.
    WER = (substitutions + deletions + insertions) / total words
    Lower is better. 0% = perfect transcription.
    """
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 padding back to pad token ID
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # Decode predictions and labels to text
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

# ── Load model ────────────────────────────────────────────────────────────
print("Loading Whisper-tiny...")
model = WhisperForConditionalGeneration.from_pretrained("./whisper_base")

# Tell the model to transcribe (not translate) in Chichewa
model.generation_config.language = "chichewa"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
print(f"Model loaded. Parameters: {model.num_parameters():,}")

# ── Subset for demo ───────────────────────────────────────────────────────
train_data = common_voice["train"]
if TRAIN_SUBSET:
    train_data = train_data.select(range(min(TRAIN_SUBSET, len(train_data))))
    print(f"Using {len(train_data)} training examples (subset for demo)")
else:
    print(f"Using full training set: {len(train_data)} examples")

# ── Training arguments ────────────────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=5,
    predict_with_generate=True,   # needed for WER calculation
    generation_max_length=225,    # max tokens Whisper generates
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,      # lower WER is better
    use_cpu=True,
    report_to="none",
)

# ── Trainer ───────────────────────────────────────────────────────────────
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=common_voice["validation"].select(range(min(20, len(common_voice["validation"])))),
    processing_class=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# ── Train ─────────────────────────────────────────────────────────────────
print(f"\nStarting training: {NUM_EPOCHS} epoch(s) on {len(train_data)} examples...")
print("Watch the loss decrease and WER improve.\n")
trainer.train()

# Save
trainer.save_model(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print(f"\n✅ Model saved to {SAVE_DIR}/")

## Cell 5 — Evaluate & Run Inference

Now we test the model.

**Word Error Rate (WER) explained:**
WER counts how many words the model gets wrong as a percentage of total words.
- WER = 0% → perfect transcription
- WER = 50% → half the words are wrong
- WER = 100% → completely wrong

After 1 epoch on 50 examples, expect WER = 80-100%. That's expected — the model has barely learned anything yet. With the full dataset and 3+ epochs, WER should drop significantly.

**Inference** means running the trained model on new audio to make predictions. We pass an audio waveform to the model and it outputs the transcribed text.

> **Note:** For inference on your own audio files, the audio must be 16kHz mono WAV or MP3. The pipeline handles resampling automatically.

In [ ]:
# ── CELL 5: Evaluate & Run Inference ──────────────────────────────────────

import gc
import torch
import numpy as np
from transformers import pipeline, WhisperForConditionalGeneration, WhisperProcessor

# Free memory before evaluation
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None
trainer.args.per_device_eval_batch_size = 2

# ── Evaluate on test set ──────────────────────────────────────────────────
print("Evaluating on test set (small subset to save memory)...")
test_subset = common_voice["test"].select(range(min(20, len(common_voice["test"]))))
results = trainer.evaluate(test_subset)
print(f"\nTest results:")
for k, v in results.items():
    print(f"  {k}: {v:.2f}" if isinstance(v, float) else f"  {k}: {v}")

# ── Run inference on test examples ───────────────────────────────────────
print("\n" + "-" * 60)
print("Inference on test audio samples:")
print("-" * 60)

# Load saved model for inference
ft_model = WhisperForConditionalGeneration.from_pretrained(SAVE_DIR)
ft_processor = WhisperProcessor.from_pretrained(SAVE_DIR)

# Pipeline makes inference easy
asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model=ft_model,
    tokenizer=ft_processor.tokenizer,
    feature_extractor=ft_processor.feature_extractor,
    device=-1,   # -1 = CPU
)

# Test on 5 examples
test_examples = common_voice["test"].select(range(5))
for i, example in enumerate(test_examples):
    audio = example["audio"]
    reference = example["sentence"]

    # Run inference
    result = asr_pipeline(
        {"array": audio["array"], "sampling_rate": audio["sampling_rate"]}
    )
    prediction = result["text"]

    print(f"\nExample {i+1}:")
    print(f"  Reference:  {reference}")
    print(f"  Prediction: {prediction}")

print("\n" + "-" * 60)
print("Note: Low accuracy after 1 epoch on 50 examples is expected.")
print("Train on the full dataset for 3+ epochs to see real improvement.")

## Cell 6 — Experiment

Go back to **Cell 4** and try these changes, then re-run Cells 4 and 5:

| Parameter | Demo value | Try this | Expected effect |
|-----------|------------|----------|-----------------|
| `TRAIN_SUBSET` | 50 | 200 or `None` | Lower WER — more data helps |
| `NUM_EPOCHS` | 1 | 3 or 5 | Lower WER — more training |
| `LEARNING_RATE` | 1e-5 | 5e-5 | Faster but may overfit |
| `BATCH_SIZE` | 4 | 2 | Less RAM if memory issues |

**Questions to explore:**
- Does WER improve significantly with more training data?
- At what point does training more epochs stop helping?
- Which Chichewa words does the model get right most consistently?
- How does the fine-tuned model compare to zero-shot Whisper (no fine-tuning)?

**Compare with zero-shot Whisper:**
Run this to see how Whisper performs without any fine-tuning:

```python
zero_shot = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny",
    device=-1,
)
# Run on the same test examples and compare WER
```

---

## What next?

- **Train on the full dataset** — remove `TRAIN_SUBSET` limit and train for 3-5 epochs overnight
- **Try a larger model** — `openai/whisper-base` (74MB) or `openai/whisper-small` (244MB) for better accuracy
- **Transcribe your own audio** — record a Chichewa sentence and run it through the pipeline
- **Push to HuggingFace Hub** — `trainer.push_to_hub('your-username/chichewa-asr')`
- **Use your own recordings** — replace Common Voice with your own audio dataset

In [ ]:
# ── CELL 6: Summary ───────────────────────────────────────────────────────

print("═" * 60)
print("TRAINING SUMMARY")
print("═" * 60)
print(f"Model:          {MODEL_NAME}")
print(f"Dataset:        Mozilla Common Voice 17.0 — Chichewa (ny)")
print(f"Training examples used: {len(train_data)}")
print(f"Epochs:         {NUM_EPOCHS}")
print(f"Learning rate:  {LEARNING_RATE}")
print(f"Batch size:     {BATCH_SIZE}")
print(f"Model saved to: {SAVE_DIR}/")
print("═" * 60)
print("\nTo transcribe your own audio:")
print('''
from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model="./asr_model", device=-1)
result = asr("your_audio.wav")
print(result["text"])
''')